# Analytics Website Code Walkthrough

This notebook provides a detailed walkthrough of the entire Analytics Website codebase. We'll examine each component in detail and understand how everything works together.

## Project Overview

The Analytics Website is a Django-based web application that:
1. Allows users to upload data files (CSV, JSON, SQL)
2. Processes these files in the background
3. Generates statistical analysis
4. Displays results in a user-friendly format

Let's start by understanding each component.

## 1. Project Structure

First, let's understand how the project is organized:

```
analytics_project/          # Main project directory
├── manage.py              # Django management script
├── analytics_project/     # Project settings
│   ├── settings.py       # Project configuration
│   ├── urls.py          # Main URL routing
│   └── celery.py        # Background task config
└── analytics_app/        # Main application
    ├── models.py        # Database models
    ├── views.py         # View functions
    ├── tasks.py         # Background tasks
    └── templates/       # HTML templates
```

Let's examine each key file and its purpose.

In [ ]:
# Let's look at key settings first
import os

# Example settings similar to your project
SETTINGS = {
    'DEBUG': True,
    'ALLOWED_FILE_TYPES': ['csv', 'json', 'sql'],
    'MAX_UPLOAD_SIZE': 30 * 1024 * 1024,  # 30MB
    'MEDIA_URL': '/uploads/',
    'MEDIA_ROOT': 'uploads/'
}

print("Allowed file types:", SETTINGS['ALLOWED_FILE_TYPES'])
print("Maximum upload size:", SETTINGS['MAX_UPLOAD_SIZE'] / (1024 * 1024), "MB")
print("File storage location:", SETTINGS['MEDIA_ROOT'])

## 2. Models and Database Structure

Let's examine our database models. The project has two main models:
1. `UploadedFile`: Stores information about uploaded files
2. `ProcessedData`: Stores analysis results

Here's how they work:

In [ ]:
# Example model structure
class UploadedFile:
    """Simulated UploadedFile model"""
    def __init__(self, user, file_path, file_type='csv'):
        self.user = user
        self.file = file_path
        self.file_type = file_type
        self.uploaded_at = 'now'
        self.processed = False
        self.error_message = None
        self.size = 0

class ProcessedData:
    """Simulated ProcessedData model"""
    def __init__(self, uploaded_file, column_name, value):
        self.uploaded_file = uploaded_file
        self.column_name = column_name
        self.value = value
        self.stats = {}

# Example usage
test_file = UploadedFile(
    user="test_user",
    file_path="example.csv",
    file_type="csv"
)

print("File details:")
print(f"Type: {test_file.file_type}")
print(f"Processed: {test_file.processed}")
print(f"Error message: {test_file.error_message}")

## 3. File Upload and Processing Flow

Let's walk through the entire process of file upload and processing:

1. User uploads a file
2. File is validated
3. Background task is queued
4. File is processed
5. Results are stored

Here's how each step works:

In [ ]:
def validate_file(file_path, file_type):
    """
    Simulate file validation
    Args:
        file_path: Path to the file
        file_type: Expected file type
    Returns:
        tuple: (is_valid, error_message)
    """
    # Check file type
    if not file_path.endswith(file_type):
        return False, f"Invalid file type. Expected {file_type}"
    
    # Simulate size check
    file_size = 1024 * 1024  # 1MB
    if file_size > SETTINGS['MAX_UPLOAD_SIZE']:
        return False, "File too large"
    
    return True, None

def process_file(file_path, file_type):
    """
    Simulate file processing
    Args:
        file_path: Path to the file
        file_type: Type of file
    Returns:
        dict: Processing results
    """
    # Simulate processing results
    results = {
        'num_rows': 1000,
        'columns': ['id', 'name', 'value'],
        'stats': {
            'value': {
                'mean': 50,
                'median': 45,
                'std': 10
            }
        }
    }
    return results

# Example usage
file_path = "example.csv"
file_type = "csv"

# Step 1: Validate
is_valid, error = validate_file(file_path, file_type)
print(f"File validation: {'Success' if is_valid else 'Failed'}")
if error:
    print(f"Error: {error}")

# Step 2: Process
if is_valid:
    results = process_file(file_path, file_type)
    print("\nProcessing results:")
    print(f"Rows: {results['num_rows']}")
    print(f"Columns: {', '.join(results['columns'])}")
    print(f"Stats: {results['stats']}")

## 4. Views and URL Routing

The project uses several key views to handle different functions:
1. Upload view for file submission
2. Results view for displaying analysis
3. Admin dashboard for oversight
4. User management views

Let's examine how these work:

In [ ]:
# Simulate view functions
class Request:
    """Simulate HTTP request"""
    def __init__(self, method='GET', user=None, files=None):
        self.method = method
        self.user = user
        self.FILES = files or {}

def upload_view(request):
    """Simulate upload view"""
    if request.method != 'POST':
        return {'template': 'upload.html'}
    
    uploaded_file = request.FILES.get('file')
    if not uploaded_file:
        return {'template': 'upload.html', 'error': 'No file uploaded'}
    
    # Create file record
    file_obj = UploadedFile(
        user=request.user,
        file_path=uploaded_file,
        file_type='csv'
    )
    
    return {'redirect': f'/result/{123}/'}

def result_view(request, file_id):
    """Simulate result view"""
    # Get file object (simulated)
    file_obj = UploadedFile(
        user="test_user",
        file_path="example.csv"
    )
    
    # Get results (simulated)
    results = [{
        'column': 'value',
        'stats': {
            'mean': 50,
            'median': 45
        }
    }]
    
    return {
        'template': 'result.html',
        'context': {
            'file': file_obj,
            'results': results
        }
    }

# Example usage
print("Upload GET request:")
req = Request(method='GET')
response = upload_view(req)
print(f"Template: {response['template']}\n")

print("Upload POST request:")
req = Request(
    method='POST',
    user="test_user",
    files={'file': 'example.csv'}
)
response = upload_view(req)
print(f"Redirect to: {response['redirect']}\n")

print("Result view:")
req = Request(user="test_user")
response = result_view(req, 123)
print(f"Template: {response['template']}")
print(f"Context: {response['context']}")

## 5. Templates and Frontend

The project uses Django templates with Bootstrap for styling. Here are the key templates:
1. base.html - Base template with common structure
2. upload.html - File upload form
3. result.html - Display processing results
4. admin_dashboard.html - Administrative interface

Let's look at how they work:

In [ ]:
<!-- Example base template structure -->
<!DOCTYPE html>
<html>
<head>
    <title>{% block title %}Analytics Website{% endblock %}</title>
    <link rel="stylesheet" href="bootstrap.min.css">
    <link rel="stylesheet" href="custom.css">
</head>
<body>
    <nav class="navbar navbar-expand-lg navbar-dark bg-primary">
        <div class="container">
            <a class="navbar-brand" href="/">Analytics Website</a>
            <!-- Navigation items -->
        </div>
    </nav>

    <main class="container my-4">
        {% block content %}{% endblock %}
    </main>
</body>
</html>

<!-- Example upload form -->
<form method="post" enctype="multipart/form-data" class="upload-form">
    {% csrf_token %}
    <div class="form-group">
        <label>Select File</label>
        <input type="file" name="file" accept=".csv,.json,.sql">
    </div>
    <button type="submit" class="btn btn-primary">Upload</button>
</form>

<!-- Example results display -->
<div class="results-container">
    <h2>Processing Results</h2>
    {% for result in results %}
        <div class="card mb-3">
            <div class="card-body">
                <h5>{{ result.column_name }}</h5>
                <p>Mean: {{ result.stats.mean }}</p>
                <p>Median: {{ result.stats.median }}</p>
            </div>
        </div>
    {% endfor %}
</div>

Let's analyze the key frontend components and patterns used in the analytics website:

1. **Template Inheritance**
- `base.html` serves as the main template
- Other templates extend it using `{% extends 'analytics_app/base.html' %}`
- Common elements (navigation, footer) are defined once
- Content blocks allow page-specific customization

2. **Bootstrap Integration**
- Bootstrap provides responsive design
- Custom CSS extends Bootstrap's classes
- Components like cards, forms, and alerts are styled consistently

3. **Form Handling**
- File upload form with proper encoding
- CSRF protection enabled
- Client-side file type validation
- Bootstrap form styling classes

4. **Results Display**
- Card-based layout for data presentation
- Dynamic rendering of statistics
- Bootstrap grid system for responsive layout
- Template filters for data formatting

5. **Static Files**
- CSS organized in separate files (base.css, components.css, layout.css)
- JavaScript for interactive features
- Proper static file serving configuration

In [ ]:
# Example of template context preparation in a view
from django.shortcuts import render
from .models import ProcessedData

def results_view(request, file_id):
    # Get processed data for the file
    results = ProcessedData.objects.filter(uploaded_file_id=file_id)
    
    # Prepare context with statistics
    context = {
        'results': [{
            'column_name': result.column_name,
            'stats': {
                'mean': result.mean,
                'median': result.median,
                'std_dev': result.std_dev,
                'min': result.min_value,
                'max': result.max_value
            }
        } for result in results]
    }
    
    return render(request, 'analytics_app/result.html', context)

# Example of template filter
from django import template
register = template.Library()

@register.filter
def format_number(value):
    """Format number with thousand separators and 2 decimal places"""
    try:
        return "{:,.2f}".format(float(value))
    except (ValueError, TypeError):
        return value

In [ ]:
/* Example custom styling */

/* Layout */
.upload-form {
    max-width: 600px;
    margin: 2rem auto;
    padding: 1.5rem;
    border-radius: 8px;
    box-shadow: 0 2px 4px rgba(0,0,0,0.1);
}

/* Components */
.results-container {
    margin-top: 2rem;
}

.results-container .card {
    transition: transform 0.2s;
}

.results-container .card:hover {
    transform: translateY(-2px);
}

/* Responsive adjustments */
@media (max-width: 768px) {
    .upload-form {
        margin: 1rem;
        padding: 1rem;
    }
    
    .results-container .card {
        margin: 0.5rem;
    }
}

In [ ]:
// Example frontend interactivity

// File upload validation
document.querySelector('.upload-form input[type="file"]').addEventListener('change', function(e) {
    const file = e.target.files[0];
    const allowedTypes = ['.csv', '.json', '.sql'];
    const fileExtension = '.' + file.name.split('.').pop().toLowerCase();
    
    if (!allowedTypes.includes(fileExtension)) {
        alert('Invalid file type. Please upload CSV, JSON, or SQL files only.');
        this.value = '';
    }
});

// Dynamic results loading
function loadResults(fileId) {
    fetch(`/api/results/${fileId}`)
        .then(response => response.json())
        .then(data => {
            const container = document.querySelector('.results-container');
            data.forEach(result => {
                const card = createResultCard(result);
                container.appendChild(card);
            });
        })
        .catch(error => console.error('Error loading results:', error));
}

// Helper function to create result cards
function createResultCard(result) {
    const card = document.createElement('div');
    card.className = 'card mb-3';
    card.innerHTML = `
        <div class="card-body">
            <h5>${result.column_name}</h5>
            <p>Mean: ${formatNumber(result.stats.mean)}</p>
            <p>Median: ${formatNumber(result.stats.median)}</p>
        </div>
    `;
    return card;
}

// Number formatting helper
function formatNumber(value) {
    return new Intl.NumberFormat('en-US', {
        minimumFractionDigits: 2,
        maximumFractionDigits: 2
    }).format(value);
}

## 6. Testing and Quality Assurance

Testing is a crucial part of maintaining the analytics website. Here's an overview of the testing approach:

1. **Unit Tests**
- Test individual components in isolation
- Verify model methods and data validation
- Check view logic and template rendering
- Validate file processing functions

2. **Integration Tests**
- Test interactions between components
- Verify file upload and processing flow
- Check database operations
- Test celery task execution

3. **End-to-End Tests**
- Test complete user workflows
- Verify frontend-backend integration
- Check responsive design
- Test error handling

4. **Testing Tools**
- Django's TestCase class
- pytest for advanced testing features
- Coverage.py for code coverage
- Browser automation for E2E tests

In [ ]:
# Example test cases
from django.test import TestCase, Client
from django.core.files.uploadedfile import SimpleUploadedFile
from .models import UploadedFile, ProcessedData
from .tasks import process_file

class FileUploadTests(TestCase):
    def setUp(self):
        self.client = Client()
        self.test_file = SimpleUploadedFile(
            "test.csv",
            b"column1,column2\n1,2\n3,4",
            content_type="text/csv"
        )
    
    def test_file_upload(self):
        # Test file upload view
        response = self.client.post('/upload/', {
            'file': self.test_file
        })
        self.assertEqual(response.status_code, 302)  # Redirect after success
        self.assertTrue(UploadedFile.objects.exists())
    
    def test_file_processing(self):
        # Test file processing task
        uploaded_file = UploadedFile.objects.create(
            file=self.test_file,
            file_type='csv'
        )
        process_file(uploaded_file.id)
        
        # Check processed results
        results = ProcessedData.objects.filter(uploaded_file=uploaded_file)
        self.assertEqual(results.count(), 2)  # Two columns
        
        # Verify statistics
        column1_stats = results.get(column_name='column1')
        self.assertEqual(column1_stats.mean, 2.0)
        self.assertEqual(column1_stats.median, 2.0)

## 7. Deployment and Production Considerations

When deploying the analytics website to production, several key aspects need to be considered:

1. **Environment Configuration**
- Use environment variables for sensitive data
- Separate development and production settings
- Configure proper security settings
- Set up logging and monitoring

2. **Database Setup**
- Use production-grade database (PostgreSQL)
- Configure database connection pools
- Set up regular backups
- Implement database migrations strategy

3. **Static Files**
- Configure static file serving
- Set up CDN for better performance
- Implement cache headers
- Minify and compress assets

4. **Security Measures**
- Enable HTTPS
- Configure CORS properly
- Implement rate limiting
- Set up proper authentication
- Regular security updates

5. **Performance Optimization**
- Enable caching layers
- Configure Celery workers
- Optimize database queries
- Monitor resource usage
- Load testing and optimization

6. **Monitoring and Maintenance**
- Set up error tracking
- Configure performance monitoring
- Regular backup verification
- Automated health checks
- Incident response plan

In [ ]:
# Example production settings
import os
from celery.schedules import crontab

# Security settings
SECURE_SSL_REDIRECT = True
SECURE_HSTS_SECONDS = 31536000
SECURE_HSTS_INCLUDE_SUBDOMAINS = True
SECURE_HSTS_PRELOAD = True
SESSION_COOKIE_SECURE = True
CSRF_COOKIE_SECURE = True

# Database configuration
DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        'NAME': os.environ.get('DB_NAME'),
        'USER': os.environ.get('DB_USER'),
        'PASSWORD': os.environ.get('DB_PASSWORD'),
        'HOST': os.environ.get('DB_HOST'),
        'PORT': os.environ.get('DB_PORT', '5432'),
        'CONN_MAX_AGE': 600,
    }
}

# Celery configuration
CELERY_BROKER_URL = os.environ.get('REDIS_URL')
CELERY_RESULT_BACKEND = os.environ.get('REDIS_URL')
CELERY_BEAT_SCHEDULE = {
    'cleanup_old_files': {
        'task': 'analytics_app.tasks.cleanup_old_files',
        'schedule': crontab(hour=0, minute=0)  # Run daily at midnight
    }
}

# Cache configuration
CACHES = {
    'default': {
        'BACKEND': 'django.core.cache.backends.redis.RedisCache',
        'LOCATION': os.environ.get('REDIS_URL'),
    }
}

# Logging configuration
LOGGING = {
    'version': 1,
    'disable_existing_loggers': False,
    'handlers': {
        'file': {
            'level': 'ERROR',
            'class': 'logging.FileHandler',
            'filename': '/var/log/django/error.log',
        },
    },
    'loggers': {
        'django': {
            'handlers': ['file'],
            'level': 'ERROR',
            'propagate': True,
        },
    },
}

## 8. API Documentation

The analytics website provides a RESTful API for programmatic access to its functionality:

1. **Authentication**
- Token-based authentication
- OAuth2 support (optional)
- Rate limiting per user/token

2. **File Operations**
- `POST /api/upload/` - Upload new file
- `GET /api/files/` - List uploaded files
- `GET /api/files/{id}/` - Get file details
- `DELETE /api/files/{id}/` - Delete file

3. **Processing**
- `GET /api/results/{file_id}/` - Get processing results
- `GET /api/status/{file_id}/` - Check processing status
- `POST /api/reprocess/{file_id}/` - Trigger reprocessing

4. **User Management**
- `POST /api/token/` - Get authentication token
- `GET /api/user/profile/` - Get user profile
- `PUT /api/user/profile/` - Update user profile

5. **Response Formats**
- JSON response structure
- Error handling
- Pagination
- Filtering and sorting

In [ ]:
# Example API views
from rest_framework import viewsets, permissions
from rest_framework.response import Response
from rest_framework.decorators import action
from .serializers import FileSerializer, ResultSerializer
from .models import UploadedFile, ProcessedData

class FileViewSet(viewsets.ModelViewSet):
    """
    API endpoint for managing file uploads
    """
    queryset = UploadedFile.objects.all()
    serializer_class = FileSerializer
    permission_classes = [permissions.IsAuthenticated]
    
    def perform_create(self, serializer):
        # Set user when creating new file
        serializer.save(user=self.request.user)
    
    @action(detail=True, methods=['get'])
    def results(self, request, pk=None):
        """Get processing results for a file"""
        file = self.get_object()
        results = ProcessedData.objects.filter(uploaded_file=file)
        serializer = ResultSerializer(results, many=True)
        return Response(serializer.data)
    
    @action(detail=True, methods=['post'])
    def reprocess(self, request, pk=None):
        """Trigger file reprocessing"""
        file = self.get_object()
        process_file.delay(file.id)
        return Response({'status': 'Processing started'})

# Example API serializers
from rest_framework import serializers

class FileSerializer(serializers.ModelSerializer):
    class Meta:
        model = UploadedFile
        fields = ['id', 'file', 'file_type', 'uploaded_at', 'status']
        read_only_fields = ['uploaded_at', 'status']

class ResultSerializer(serializers.ModelSerializer):
    stats = serializers.SerializerMethodField()
    
    class Meta:
        model = ProcessedData
        fields = ['column_name', 'stats']
    
    def get_stats(self, obj):
        return {
            'mean': obj.mean,
            'median': obj.median,
            'std_dev': obj.std_dev,
            'min': obj.min_value,
            'max': obj.max_value
        }

# Analytics Website Tutorial

This notebook provides a hands-on guide to understanding and working with the Analytics Website project. We'll explore the main components and demonstrate how they work together.

## 1. Project Setup and Dependencies

First, let's look at the project's required dependencies and how to set them up. Our project uses:
- Django for the web framework
- Celery for background tasks
- pandas for data processing
- Various other Python packages

Let's see how to set up the development environment.

In [ ]:
# Example of installing requirements
!pip install -r requirements.txt

# Import key libraries
import os
import django
import pandas as pd
import json

# This would be needed in a real environment, but we'll skip actual execution
# os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'analytics_project.settings')
# django.setup()

## 2. File Upload System

Our website handles file uploads with several key features:
- File type validation
- Size limits
- Storage management
- Processing queue

Let's look at how the upload system works:

In [ ]:
# Example of file validation logic
def validate_file(file_path):
    """Demonstrate file validation logic"""
    
    # Check file extension
    allowed_types = ['csv', 'json', 'sql']
    ext = file_path.split('.')[-1].lower()
    
    if ext not in allowed_types:
        return False, f"Invalid file type. Allowed types: {', '.join(allowed_types)}"
    
    # Check file size (example: 30MB limit)
    max_size = 30 * 1024 * 1024  # 30MB in bytes
    file_size = os.path.getsize(file_path)
    
    if file_size > max_size:
        return False, f"File too large. Maximum size is {max_size/(1024*1024)}MB"
    
    return True, "File is valid"

# Example usage
test_file = "example.csv"
is_valid, message = validate_file(test_file)
print(f"Validation result: {message}")

## 3. Data Processing System

Once a file is uploaded, our system processes it using Celery tasks. Here's how the processing pipeline works:
1. File is uploaded and validated
2. Processing task is queued
3. Data is analyzed
4. Results are stored

Let's look at an example of data processing:

In [ ]:
# Example of data processing
def process_csv_file(file_path):
    """Demonstrate CSV file processing"""
    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Calculate basic statistics
    stats = {
        'row_count': len(df),
        'column_count': len(df.columns),
        'numeric_columns': df.select_dtypes(include=['int64', 'float64']).columns.tolist(),
        'missing_values': df.isnull().sum().to_dict()
    }
    
    # Calculate statistics for numeric columns
    numeric_stats = {}
    for col in stats['numeric_columns']:
        numeric_stats[col] = {
            'mean': df[col].mean(),
            'median': df[col].median(),
            'std': df[col].std(),
            'min': df[col].min(),
            'max': df[col].max()
        }
    
    stats['numeric_stats'] = numeric_stats
    return stats

# Example usage (not actually run)
sample_stats = {
    'row_count': 1000,
    'column_count': 5,
    'numeric_columns': ['salary', 'age', 'experience'],
    'numeric_stats': {
        'salary': {'mean': 50000, 'median': 48000},
        'age': {'mean': 35, 'median': 33}
    }
}
print("Sample processing results:", json.dumps(sample_stats, indent=2))